<a href="https://colab.research.google.com/github/AlexeyTri/SemMed_fall25/blob/main/HW/HW_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание Цель работы в создании CNN для решения задачи классификации

Задачи:
1. Загрузить выборку MNIST, входит в пакет tjrchvision.datasets. Предобработать данные
2. Построить CNN
3. Достичь уровня более 90% на тестовой выборке

In [ ]:
!pip install torchmetrics
import torchmetrics
import numpy as np
import torch
from sklearn.datasets import load_sample_images
import torchvision
import torchvision.transforms.v2 as T
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(42)

In [ ]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'


**Загрузка датасета**

!!! Внимание, torch принимает на вход тензоры размерности (n_samples, n_channels, hight, weight)

In [ ]:
# your code here

**CNN**

CNN строится на основании модулей ResNet34, с некоторыми изменениями

In [ ]:
class ResidualUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        DefaultConv2d = partial(nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.main_layer = nn.Sequential(
            DefaultConv2d(in_channels=in_channels, out_channels=out_channels, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            DefaultConv2d(in_channels=out_channels, out_channels=out_channels),
            nn.BatchNorm2d(out_channels)
            """добавьте еще 1 слой DefaultConv2d"""
            # your code here





        )
        if stride > 1:
            self.skip_connection = nn.Sequential(
                DefaultConv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1, stride=stride, padding=0),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.skip_connection = nn.Identity()
    def forward(self, inputs):
        return F.relu(self.main_layer(inputs) + self.skip_connection(inputs))

In [ ]:
class ResNet34(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
            """ внесите корректные входные параметры """
            nn.Conv2d(in_channels="""your code here""", out_channels="""your code here""", kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(num_features="""your code here"""),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)]
        prev_filters = 64
        """постройте три варианта модели: с 4-мя слоями 512, 5-тью слоями 512, 6-тью слоями 512"""
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(
                ResidualUnit(prev_filters, filters, stride=stride))
            prev_filters = filters
        layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10)
        ]

        self.resnet = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.resnet(inputs)

**обучение моделей**

Обучите полученные модели на 10-ти эпохах
оптимизатор, функцию потерь, функцию качества предсказания модели определите самостоятельно

Сформируйте ВЫВОД, какая модель и при каких критериях обучения дала наилучший результат, как вы считаете почему?

In [ ]:
# your code here

**Сфоррмируйте предсказания на тестовой выборке**

В Выоде напишите, можно ли еще улучшить предсказания модели и каким образом

In [ ]:
# your code here